# Phase 2: Chest X-Ray Computer Vision Classifier

This notebook is designed to be uploaded to **Google Colab** to utilize a free cloud GPU for training a PyTorch Convolutional Neural Network (CNN) on medical imaging data.

**Objective:** Classify Chest X-Rays as Normal vs. Pneumonia.

In [ ]:
# Install MedMNIST, a lightweight medical imaging dataset perfect for fast cloud training
!pip install -q medmnist

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import transforms, models
import medmnist
from medmnist import INFO

print(f"PyTorch version: {torch.__version__}")

# Check if Cloud GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Ingestion & Preprocessing
We use `PneumoniaMNIST`, a subset of pediatric chest X-rays. It automatically downloads and provides ready-to-train PyTorch datasets.

In [ ]:
data_flag = 'pneumoniamnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

# Preprocessing: Convert images to Tensor and normalize
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Download and load datasets
train_dataset = DataClass(split='train', transform=data_transform, download=True)
val_dataset = DataClass(split='val', transform=data_transform, download=True)

train_loader = data.DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=128, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 2. Model Architecture
We load a pre-trained ResNet18 model (Transfer Learning) and modify the final fully connected layer for binary classification.

In [ ]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Modifying the first convolution layer to accept 1-channel grayscale X-rays instead of 3-channel RGB
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Replace the final layer for 2 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

## 3. Training Loop
Running the actual optimization loop on the cloud GPU.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3

print("Starting training on Cloud GPU...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.squeeze().to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Training Loss: {running_loss/len(train_loader):.4f}")

print("Training complete! Model is ready for evaluation.")

## 4. Evaluation & Export

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.squeeze().to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

# Save the weights to be exported to the Node.js backend
torch.save(model.state_dict(), 'pneumonia_cnn_weights.pth')
print("Model saved to pneumonia_cnn_weights.pth")